In [2]:
# extract scenarios and save to JSONL file
import json

data = []
for i, scenario in enumerate(json.load(open('scenarios-labeled.json', 'r'))):
    data.append({
        'id': i + 1, 
        'text': scenario['text'], 
        'label':[] if not 'labels' in scenario else scenario['labels']
    })
                             
with open('scenarios-labeled-input.jsonl', 'w') as f:
    for d in data:
        f.write(json.dumps(d) + '\n')

In [2]:
import json

data = json.load(open('scenarios-risked.json', 'r'))

# 要排除的ID数字
exclude = [45, 65, 110, 112, 121]

updated = []
for item in data:
    # 从 scenario_id 中提取数字部分 (例如 "MAS-R-45" -> 45)
    scenario_num = int(item['scenario_id'].split('-')[-1])
    
    if scenario_num in exclude:
        continue
    
    # 数据已经有 labels 字段，直接保留
    updated.append(item)
        
json.dump(updated, open('scenarios-risked1.json', 'w'), indent=2)

print(f"原始记录数: {len(data)}")
print(f"排除后记录数: {len(updated)}")


原始记录数: 198
排除后记录数: 193


In [3]:
data[175]

{'app_url': 'https://play.google.com/store/apps/details?id=com.paypal.android.p2pmobile',
 'usage_freq': 'weekly',
 'text': "This is the PayPal Wallet screen. On this screen, I can manage most of my finances through PayPal. I can add bank accounts to my PayPal, I can apply for new cards through PayPal, I can cash checks that I receive from this screen, and I can even manage my payment plans that I've used from PayPal; for example, the Pay in 4 that PayPal offers to many retailers, I can manage from this screen. I can setup other payments such as Google Pay and Samsung pay from this screen. I can manage my automatic payments I have setup for things like subscriptions, and I can apply for credit through PayPal with one of their options that are available to me. I can also create a debit card from PayPal through this screen, which will allow me to use money on my account rather than my bank card to make purchases. Finally, I can add cash for specific stores that I frequent.",
 'scenario_i

In [ ]:
last_count = 0
count = 0
for i, d in enumerate(data):
    count += len(d['text'])
    if count > 5000:
        print('%i = %i chars' % (i-1, last_count))
        break
    last_count = count
print(count)

4 = 4567 chars
5410


In [ ]:
# Test grammary checking facility
import requests, uuid

key = 'FXR0LFQ6V3QI6E38XSYITD088PZZNG2V'

url = 'https://api.sapling.ai/api/v1/edits'
post_data = {
    'key': key,
    'text': data[175]['text'],
    'session_id': uuid.uuid4().hex.upper(),
}

try:
    resp = requests.post(url, json=post_data)
    resp_json = resp.json()
    if 200 <= resp.status_code < 300:
      edits = resp_json['edits']
      print('Edits: ', edits)
    else:
      print('Error: ', resp_json)
except Exception as e:
    print('Error: ', e)

Edits:  [{'end': 25, 'error_type': 'R:VERB:SVA', 'general_error_type': 'Grammar', 'id': 'bf7bbe22-e822-5fdb-ae72-ee04a74754a2', 'replacement': 'has', 'sentence': 'This app manages and have streaks for variable habits which are very good for us to maintaining regular activities as well they discipline us in order to the data we have provided to them in the app.', 'sentence_start': 102, 'start': 21}, {'end': 95, 'error_type': 'R:VERB:FORM', 'general_error_type': 'Grammar', 'id': '1c2ba86d-338b-52ab-baba-151dab5e9f47', 'replacement': 'maintain', 'sentence': 'This app manages and have streaks for variable habits which are very good for us to maintaining regular activities as well they discipline us in order to the data we have provided to them in the app.', 'sentence_start': 102, 'start': 84}, {'end': 56, 'error_type': 'M:DET:ART', 'general_error_type': 'Grammar', 'id': '3fb1d7ce-c3b3-58ee-8f03-8f966b46f332', 'replacement': 'the', 'sentence': 'They have mentioned and promised that, they wi

In [ ]:
# extract information types from tagged words
import spacy

nlp = spacy.load("en_core_web_sm")

def find_start_char(doc, head_token):
    prefix_tags = ['NOUN', 'ADJ', 'ADP' 'NUM']
    exclusions = ['more']
    
    for i in reversed(range(0, head_token.i)):
        if not doc[i].pos_ in prefix_tags or doc[i].text in exclusions:
            return doc[i + 1].idx
    return head_token.idx

def find_end_char(doc, head_token):
    prefix_tags = ['NOUN']
    
    for i in range(head_token.i + 1, len(doc)):
        if not doc[i].pos_ in prefix_tags:
            return doc[i - 1].idx + len(doc[i - 1].text)
    return head_token.idx + len(head_token.text)

def get_token(doc, start_char, end_char):
    for token in doc:
        if token.idx >= start_char and token.idx < end_char:
            return token
    return None

def extract_types(scenario):
    extracted = {}
    doc = nlp(scenario['text'])
    missed = 0
    
    for label in scenario['label']:
        phrase = None
        match = []
        if label[2] == 'QUE':
            phrase = scenario['text'][label[0]:label[1]].lower()
            match = [label[0], label[1]]
            
        elif label[2] == 'SIM':
            token = get_token(doc, label[0], label[1])
            start = find_start_char(doc, token)
            end = find_end_char(doc, token)
            phrase = scenario['text'][start:end].lower()
            match = [start, end]

        if not phrase:
            missed += 1
            continue
        if not phrase in extracted:
            extracted[phrase] = []
        extracted[phrase].append([start, end])

    for phrase, match in extracted.items():
        print(phrase)

    print('\nFound %i/%i or %0.3f' % (
        len(scenario['label']) - missed,
        len(scenario['label']),
        (len(scenario['label']) - missed) / len(scenario['label'])
    ))
        
extract_types(data[2])

account information
information
full name
contact info
phone number
email id
identity
account ownership
control settings
password
security settings
factor authentication
logins
alerts
payment preferences
important privacy issues
preferences
news feed
reaction preferences
emojis
stickers
notification preferences
shortcuts preferences
language
region
media preferences
autoplay
data saver
video quality
themes
dark mode
light mode
visibility settings
who can see my posts, stories, and profile

Found 35/37 or 0.946
